# debug

In [1]:
import requests

url = "https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
}

r = requests.get(url, headers=headers)
print("Status:", r.status_code)
print("Final URL:", r.url)  # cek kalau ada redirect
print("Length:", len(r.text))
print(r.text[:2000])  # lihat 2000 karakter pertama

Status: 200
Final URL: https://analitik.kemendag.go.id/#/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga
Length: 1340
<!doctype html><html xmlns:ng="" xmlns:tb=""><head><meta charset="UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=edge"><meta name="viewport" content="initial-scale=1,maximum-scale=2,width=device-width,height=device-height,viewport-fit=cover"><meta name="format-detection" content="telephone=no"><meta name="vizportal-config" data-buildid="2025_1_43_onhrvsr8b1o" data-staticassetsurlprefix=""><link href="vendors-vizportal.css?5e26def90e0996e201db" rel="stylesheet"><link href="vizportal.css?5e26def90e0996e201db" rel="stylesheet"><script src="/javascripts/api/tableau-2.min.js?5e26def90e0996e201db"></script><script src="jquery.min.js?5e26def90e0996e201db"></script><script src="rsa.js?5e26def90e0996e201db"></script><script src="underscore-min.js?5e26def90e0996e201db"></script><script src="q.min.js?5e26def90e0996e201db"></script><script src="canvas-to-blob.min.js?5e

In [2]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")
container = soup.find("textarea", {"id": "tsConfigContainer"})
print("Container ditemukan?", container is not None)
if container:
    print("Isi (500 char pertama):", container.text[:500])

Container ditemukan? False


In [3]:
import requests

s = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Referer": "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
}

# 1. Kunjungi halaman SP2KP dulu biar dapat cookie awal
s.get("https://sp2kp.kemendag.go.id/statistik/tabulasi-harga", headers=headers)

# 2. Baru hit URL embed Tableau, pakai session yang sama + Referer
url = "https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga?:embed=yes&:showShareOptions=false&:showAskData=false&:customViews=no"
r = s.get(url, headers=headers)

print("Status:", r.status_code)
print("Final URL:", r.url)
print("Length:", len(r.text))
print(r.text[:1500])

Status: 200
Final URL: https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga?:embed=yes&:showShareOptions=false&:showAskData=false&:customViews=no
Length: 5238
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8" />
    <meta http-equiv="Expires" content="Thu, 01 Jan 1970 00:00:00 GMT" />
    <meta http-equiv="Pragma" content="no-cache" />
    <meta name="format-detection" content="telephone=no" />
    <meta name="viewport" content="width=device-width, maximum-scale=2.0" />              <title>Tableau</title>
    <style type="text/css">
    html {
     height: 100%;
    }
    body {
     position: absolute;
     left: 0;
     right: 0;
     bottom: 0;
     top: 0;
     margin: 0;
     padding: 0;
    }
    #tabBootErr {
     display: none;
     position: absolute;
     top: 100px;
     width: 300px;
     left: 50%;
     margin-left: -150px;
     border: 1px solid black;
     background: white;
     padding: 0;
     box-shadow: 0px 5px 10px #adadad;
  

In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")
container = soup.find("textarea", {"id": "tsConfigContainer"})
print("Container ditemukan?", container is not None)

if container:
    print(container.text[:1000])
else:
    # kalau masih gak ketemu, print full text buat lihat strukturnya
    print(r.text[1500:5238])

Container ditemukan? True



In [5]:
from tableauscraper import TableauScraper as TS

ts = TS()
ts.session = s  # pakai session yang sudah punya cookie + referer dari langkah sebelumnya
ts.loads(url)

workbook = ts.getWorkbook()
for sheet in workbook.worksheets:
    print(sheet.name, "->", sheet.data.shape)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
import requests
from tableauscraper import TableauScraper as TS

s = requests.Session()

# nempelin header PERMANEN ke session, bukan per-request
s.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Referer": "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
})

# kunjungi halaman SP2KP dulu buat dapat cookie
s.get("https://sp2kp.kemendag.go.id/statistik/tabulasi-harga")

# baru load pakai tableauscraper, session udah bawa header otomatis
url = "https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga?:embed=yes&:showShareOptions=false&:showAskData=false&:customViews=no"

ts = TS()
ts.session = s
ts.loads(url)

workbook = ts.getWorkbook()
for sheet in workbook.worksheets:
    print(sheet.name, "->", sheet.data.shape)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
import requests
from tableauscraper import TableauScraper as TS

s = requests.Session()
s.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Referer": "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
})

s.get("https://sp2kp.kemendag.go.id/statistik/tabulasi-harga")

# PENTING: pakai URL bersih, TANPA query string di belakangnya
url = "https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga"

ts = TS()
ts.session = s
ts.loads(url)

workbook = ts.getWorkbook()
for sheet in workbook.worksheets:
    print(sheet.name, "->", sheet.data.shape)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
from tableauscraper import api

# panggil PERSIS fungsi yang sama yang dipakai ts.loads() di dalam
raw = api.getTableauVizForSession(ts, s, url)

print("Length:", len(raw))
print(raw[:3000])

Length: 5238
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8" />
    <meta http-equiv="Expires" content="Thu, 01 Jan 1970 00:00:00 GMT" />
    <meta http-equiv="Pragma" content="no-cache" />
    <meta name="format-detection" content="telephone=no" />
    <meta name="viewport" content="width=device-width, maximum-scale=2.0" />              <title>Tableau</title>
    <style type="text/css">
    html {
     height: 100%;
    }
    body {
     position: absolute;
     left: 0;
     right: 0;
     bottom: 0;
     top: 0;
     margin: 0;
     padding: 0;
    }
    #tabBootErr {
     display: none;
     position: absolute;
     top: 100px;
     width: 300px;
     left: 50%;
     margin-left: -150px;
     border: 1px solid black;
     background: white;
     padding: 0;
     box-shadow: 0px 5px 10px #adadad;
     font-family: Arial, Helvetica, sans-serif;
     z-index:999;
    }
    #tabBootErrTitle {
     font-size: 11pt;
     font-weight: bold;
     color: #1F437D;
     background-col

In [ ]:
print(raw[3000:5238])

       opacity: 0.24;
      }
    </style>

    <div id="initializing_thin_client" class="wcBody">
        <div id="loadingSpinner" style="display:none">
          <div id="svg-spinner-container">
            <svg id="svg-spinner" viewBox="0 0 50 50"><style>#tail{fill:url(#fade)}#head{fill:#616570}stop{stop-color:#616570}</style><linearGradient id="fade" x2="50" y1="25" y2="25" gradientUnits="userSpaceOnUse"><stop offset="0" stop-opacity="0"/><stop offset=".15" stop-opacity=".04"/><stop offset=".3" stop-opacity=".16"/><stop offset=".45" stop-opacity=".36"/><stop offset=".61" stop-opacity=".64"/><stop offset=".76"/></linearGradient><path id="head" d="M0 25a25 25 0 1 0 50 0h-3.9a21.1 21.1 0 1 1-42.2 0" /><path id="tail" d="M50 25a25 25 0 0 0-50 0h3.9a21.1 21.1 0 1 1 42.2 0" /></svg>
          </div>
        </div>
        <div class="wcGlassPane" style="display:none" id="loadingGlassPane"></div>
    </div>
    <div id="centeringContainer">
        <div class="wcStaticImage" id="staticIma

In [ ]:
import asyncio
import threading
import re
import json
from playwright.async_api import async_playwright

captured = {}

async def handle_response(response):
    if "bootstrapSession" in response.url:
        try:
            captured["raw"] = await response.text()
        except Exception as e:
            print("gagal baca response:", e)

url = "https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga?:embed=yes&:showShareOptions=false&:showAskData=false&:customViews=no"

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
        )
        page = await context.new_page()
        page.on("response", handle_response)

        await page.goto("https://sp2kp.kemendag.go.id/statistik/tabulasi-harga")
        await page.goto(url, referer="https://sp2kp.kemendag.go.id/statistik/tabulasi-harga")
        await page.wait_for_timeout(8000)

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        finally:
            loop.close()

    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("Berhasil dapat data?", "raw" in captured)
if "raw" in captured:
    print(len(captured["raw"]))

Berhasil dapat data? False


In [ ]:
import asyncio
import threading
from playwright.async_api import async_playwright

all_vizql_urls = []
page_errors = []

async def handle_response(response):
    if "vizql" in response.url.lower() or "session" in response.url.lower():
        all_vizql_urls.append((response.status, response.url))

url = "https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga?:embed=yes&:showShareOptions=false&:showAskData=false&:customViews=no"

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
        )
        page = await context.new_page()
        page.on("response", handle_response)
        page.on("console", lambda msg: page_errors.append(f"CONSOLE: {msg.text}"))
        page.on("pageerror", lambda exc: page_errors.append(f"PAGEERROR: {exc}"))

        await page.goto("https://sp2kp.kemendag.go.id/statistik/tabulasi-harga")
        await page.goto(url, referer="https://sp2kp.kemendag.go.id/statistik/tabulasi-harga")
        await page.wait_for_timeout(10000)

        # ambil screenshot buat lihat kondisi akhir halaman
        await page.screenshot(path="debug_screenshot.png", full_page=True)

        # cek isi tsConfigContainer di top-level page maupun iframe
        content = await page.content()
        with open("debug_page.html", "w", encoding="utf-8") as f:
            f.write(content)

        frames_info = [(f.url) for f in page.frames]
        print("Frames:", frames_info)

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        finally:
            loop.close()

    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("\n=== VIZQL/SESSION URLs ===")
for status, u in all_vizql_urls:
    print(status, u)

print("\n=== PAGE ERRORS ===")
for e in page_errors[:20]:
    print(e)

Frames: ['https://analitik.kemendag.go.id/#/signin?externalRedirect=%2Fviews%2FTabulasiHargaSP2KP_17708080966730%2FTabulasiHarga%3F%3Aembed%3Dyes%26%3AshowShareOptions%3Dfalse%26%3AshowAskData%3Dfalse%26%3AcustomViews%3Dno%26%3Aredirect%3Dauth&idpConfigurationId=&isDefaultIdentityPoolLogin=true&site=']

=== VIZQL/SESSION URLs ===
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/built-dojo/tableau/web/css/tableau.css
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/PreBootstrap.min.js
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/mscorlib.min.js
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/css/vqlweb.css
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/jquery.min.js
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/ViewerBootstrap.js
401 https://analitik.kemendag.go.id/vizql/w/TabulasiHargaSP2KP_17708080966730/v/TabulasiHarga/startSession/viewing?%3Aembed=y

In [ ]:
import asyncio
import threading
from playwright.async_api import async_playwright

captured = {}
all_urls = []

async def handle_response(response):
    u = response.url
    if "vizql" in u.lower() or "session" in u.lower() or "trusted" in u.lower():
        all_urls.append((response.status, u))
    if "bootstrapSession" in u:
        try:
            captured["raw"] = await response.text()
        except Exception as e:
            print("gagal baca response:", e)

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
        )
        page = await context.new_page()
        page.on("response", handle_response)

        # HANYA buka halaman SP2KP asli, JANGAN navigasi manual ke URL Tableau
        await page.goto("https://sp2kp.kemendag.go.id/statistik/tabulasi-harga", wait_until="networkidle", timeout=30000)
        await page.wait_for_timeout(10000)

        await page.screenshot(path="debug_screenshot2.png", full_page=True)
        frames_info = [f.url for f in page.frames]
        print("Frames:", frames_info)

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("\n=== URLs ===")
for status, u in all_urls:
    print(status, u)

print("\nBerhasil dapat bootstrap?", "raw" in captured)

Exception in thread Thread-7 (runner):
Traceback (most recent call last):
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Aufii\AppData\Local\Temp\ipykernel_12428\1583465344.py", line 43, in runner
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\asyncio\base_events.py", line 687, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "C:\Users\Aufii\AppData\Local\Temp\ipykernel_12428\1583465344.py", line 28, in run
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\site-packages\playwright\async_api\_generated.py", line 8992, 


=== URLs ===
302 https://analitik.kemendag.go.id/trusted/1Cf5qg8iSK2WN4OmrUfbLw==:FLLRE9a7CjpO0pkcDd8ydfUT/views/TabulasiHargaSP2KP_17708080966730/TabulasiHarga?:embed=yes&:showShareOptions=false&:showAskData=false&:customViews=no
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/built-dojo/tableau/web/css/tableau.css
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/css/vqlweb.css
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/PreBootstrap.min.js
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/mscorlib.min.js
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/ViewerBootstrap.js
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/formatters-and-parsers.en_US.js
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/jsstrings_en.js
200 https://analitik.kemendag.go.id/vizql/v_202512603060927/javascripts/messages.en_US.js
200 https://analitik.kemendag

In [ ]:
import re
import json
from tableauscraper import TableauScraper as TS
from tableauscraper import dashboard

raw = captured["raw"]

dataReg = re.search(r"\d+;({.*})\d+;({.*})", raw, re.DOTALL)
info = json.loads(dataReg.group(1))
data = json.loads(dataReg.group(2))

ts = TS()
ts.data = data
ts.info = info
ts.dashboard = info["sheetName"]

# INI YANG KELEWAT SEBELUMNYA
presModelMap = data["secondaryInfo"]["presModelMap"]
ts.dataSegments = presModelMap["dataDictionary"]["presModelHolder"]["genDataDictionaryPresModel"]["dataSegments"]

workbook = dashboard.getWorksheets(ts, ts.data, ts.info)

for sheet in workbook.worksheets:
    print(sheet.name, "->", sheet.data.shape)

KeyError: 'presModelMap'

In [ ]:
print("Top-level keys:", list(data.keys()))
print()
if "secondaryInfo" in data:
    print("secondaryInfo keys:", list(data["secondaryInfo"].keys()))

Top-level keys: ['secondaryInfo']

secondaryInfo keys: []


In [ ]:
def find_key(obj, target_key, path=""):
    results = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}"
            if k == target_key:
                results.append(new_path)
            results.extend(find_key(v, target_key, new_path))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            results.extend(find_key(v, target_key, f"{path}[{i}]"))
    return results

paths = find_key(data, "dataSegments")
print("Lokasi dataSegments ditemukan di:")
for p in paths:
    print(p)

Lokasi dataSegments ditemukan di:


In [ ]:
print("Panjang raw:", len(raw))
print()
print("200 karakter pertama:")
print(raw[:200])
print()
print("200 karakter terakhir:")
print(raw[-200:])

Panjang raw: 95153

200 karakter pertama:
95124;{"sheetName":"Tabulasi Harga   ","layoutId":"18157853140954229409","newSessionId":"C827B9E8263E42F29CDFAC4AF2DE584E-1:0","newClientNum":"1","isDeferredBootstrap":false,"workbookLocale":"id_ID","

200 karakter terakhir:
arnings": false}}},{"keyId": "doc:summary-data-changed-event","presModelHolder":{"genSummaryDataChangeEventPresModel":{"visualIdPresModel":{"worksheet": "Tabulasi SP2KP"}}}}]}}}20;{"secondaryInfo":{}}


In [ ]:
import re

# cari semua posisi yang match pola "angka;{"
matches = list(re.finditer(r'\d{2,};\{', raw))
print(f"Ditemukan {len(matches)} kemungkinan segmen")
for m in matches[:10]:
    print(m.start(), "->", raw[m.start():m.start()+50])

Ditemukan 2 kemungkinan segmen
0 -> 95124;{"sheetName":"Tabulasi Harga   ","layoutId":
95130 -> 20;{"secondaryInfo":{}}


In [ ]:
import json

# ambil hanya blok pertama, buang prefix angka+titik-koma di depannya
first_semicolon = raw.index(";")
json_str = raw[first_semicolon+1 : 95124 + first_semicolon + 1]

parsed = json.loads(json_str)
print("Top-level keys:", list(parsed.keys()))

Top-level keys: ['sheetName', 'layoutId', 'newSessionId', 'newClientNum', 'isDeferredBootstrap', 'workbookLocale', 'allowSubscriptions', 'allowSubscribeOnDataPresent', 'repositoryUrl', 'isResponseCacheHit', 'worldUpdate']


In [ ]:
def find_key(obj, target_key, path="", max_results=20):
    results = []
    if len(results) >= max_results:
        return results
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}"
            if k == target_key:
                results.append(new_path)
            results.extend(find_key(v, target_key, new_path))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            results.extend(find_key(v, target_key, f"{path}[{i}]"))
    return results

for key_name in ["vqlCmdResponse", "presModelMap", "dataSegments", "dataDictionary"]:
    found = find_key(parsed, key_name)
    print(f"\n{key_name}:")
    for p in found[:5]:
        print(" ", p)


vqlCmdResponse:

presModelMap:

dataSegments:

dataDictionary:
  .worldUpdate.applicationPresModel.dataDictionary


In [ ]:
app_model = parsed["worldUpdate"]["applicationPresModel"]
print("Keys di applicationPresModel:", list(app_model.keys()))

Keys di applicationPresModel: ['renderMode', 'workbookPresModel', 'dateFormat', 'timeFormat', 'numberFormats', 'toolbarPresModel', 'vizViewerToolbarPresModel', 'dataDictionary', 'presentationLayerNotification']


In [ ]:
dd = app_model["dataDictionary"]
print("Keys di dataDictionary:", list(dd.keys()) if isinstance(dd, dict) else type(dd))

Keys di dataDictionary: []


In [ ]:
def find_key(obj, target_key, path="", max_results=20):
    results = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}"
            if k == target_key:
                results.append(new_path)
            if len(results) < max_results:
                results.extend(find_key(v, target_key, new_path, max_results))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            if len(results) < max_results:
                results.extend(find_key(v, target_key, f"{path}[{i}]", max_results))
    return results[:max_results]

for key_name in ["workbookPresModel", "dashboardPresModel", "worksheetPresModel", "zones", "columnsData", "dataValues"]:
    found = find_key(app_model, key_name)
    print(f"\n{key_name}:")
    for p in found:
        print(" ", p)


workbookPresModel:
  .workbookPresModel

dashboardPresModel:
  .workbookPresModel.dashboardPresModel

worksheetPresModel:

zones:
  .workbookPresModel.dashboardPresModel.zones

columnsData:

dataValues:


In [ ]:
dashboard_model = app_model["workbookPresModel"]["dashboardPresModel"]
print("Keys di dashboardPresModel:", list(dashboard_model.keys()))
print()
zones = dashboard_model["zones"]
print("Tipe zones:", type(zones))
if isinstance(zones, dict):
    print("Jumlah zones:", len(zones))
    # print salah satu contoh zone buat lihat strukturnya
    first_key = list(zones.keys())[0]
    print(f"\nContoh zone '{first_key}':")
    print(json.dumps(zones[first_key], indent=2)[:1500])

Keys di dashboardPresModel: ['sheetPath', 'zones', 'activeZoneId', 'activeZoneIds', 'userActions', 'cssAttrs', 'autoUpdate', 'hasSelection', 'viewportSize', 'visualIds', 'invalidSheets', 'viewIds', 'sheetLayoutInfo', 'modifiedSheets', 'dashboardDeviceLayouts', 'autogeneratedDeviceLayouts', 'deviceLayoutInfo', 'previewBarIsVisible', 'dashboardSizePresModel', 'sizeMode', 'isCopiableZoneSelected', 'singleValueFields']

Tipe zones: <class 'dict'>
Jumlah zones: 11

Contoh zone '105':
{
  "zoneId": 105,
  "zoneCommon": {
    "zoneId": 105,
    "parentZoneId": 4,
    "zoneType": "filter",
    "x": 8,
    "y": 141,
    "w": 192,
    "h": 59,
    "contentX": 12,
    "contentY": 145,
    "contentW": 184,
    "contentH": 51,
    "canHaveTitle": false,
    "hasTitle": true,
    "name": "Provinsi",
    "zoneText": "<formatted-text>\n</formatted-text>\n",
    "styledBox": {
      "uw": 0,
      "borderStyle": "bs-none",
      "borderColor": "rgb(0,0,0)",
      "margin": 4,
      "fillColor": "rgba(0

In [ ]:
def find_key_pattern(obj, pattern, path="", max_results=15):
    results = []
    pattern_lower = pattern.lower()
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}"
            if pattern_lower in k.lower():
                results.append(new_path)
            if len(results) < max_results:
                results.extend(find_key_pattern(v, pattern, new_path, max_results))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            if len(results) < max_results:
                results.extend(find_key_pattern(v, pattern, f"{path}[{i}]", max_results))
    return results[:max_results]

for pattern in ["Segment", "resModel", "column", "Value", "worksheet"]:
    found = find_key_pattern(app_model, pattern)
    print(f"\nPola '{pattern}':")
    for p in found:
        print(" ", p)


Pola 'Segment':

Pola 'resModel':
  .workbookPresModel
  .workbookPresModel.dashboardPresModel
  .workbookPresModel.dashboardPresModel.zones.105.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.106.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.128.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.129.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.138.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.17.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.4.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.47.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.47.presModelHolder.visual.visualIdPresModel
  .workbookPresModel.dashboardPresModel.zones.48.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.67.presModelHolder
  .workbookPresModel.dashboardPresModel.zones.70.presModelHolder
  .workbookPresModel.dashboardPresModel.dashboardSizePresModel

Pola 'column':
  .workbookPresM

In [ ]:
zone_47 = dashboard_model["zones"]["47"]
visual = zone_47["presModelHolder"]["visual"]
print("Keys di visual:", list(visual.keys()))

Keys di visual: ['hasVizArtTemplate', 'hasVizExtension', 'selectionList', 'brushingSelectionList', 'annotationList', 'filtersJson', 'vizAltText', 'geometryJson', 'cacheUrlInfoJson', 'sortIndicators', 'isMap', 'hasBackgroundImage', 'floatingToolbarVisibility', 'geographicSearchVisibility', 'vizNavigationSetting', 'isLayerControlButtonAllowed', 'isLayerControlButtonEnabled', 'wholeVizScrollThreshold', 'defaultMapToolEnum', 'hasModifiedAxes', 'valid', 'visualIdPresModel', 'tooltipMode', 'bgColor', 'paneColor', 'headerColor', 'emptyHighlightFogAll', 'totalMarks', 'shouldShowStats', 'showImageLimitToast', 'selectedMarks', 'numberOfColumns', 'numberOfRows', 'lastComputedCalculation', 'animationStyleSettings', 'blockMarkAnimationForNextCommand']


In [ ]:
def find_key_pattern(obj, pattern, path="", max_results=15):
    results = []
    pattern_lower = pattern.lower()
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}"
            if pattern_lower in k.lower():
                results.append(new_path)
            if len(results) < max_results:
                results.extend(find_key_pattern(v, pattern, new_path, max_results))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            if len(results) < max_results:
                results.extend(find_key_pattern(v, pattern, f"{path}[{i}]", max_results))
    return results[:max_results]

for pattern in ["Pane", "Row", "Cell", "Table", "Dictionary", "dataValue", "aliasIndices"]:
    found = find_key_pattern(visual, pattern)
    print(f"\nPola '{pattern}':")
    for p in found:
        print(" ", p)


Pola 'Pane':
  .paneColor

Pola 'Row':
  .numberOfRows

Pola 'Cell':

Pola 'Table':

Pola 'Dictionary':

Pola 'dataValue':

Pola 'aliasIndices':


In [ ]:
zones = dashboard_model["zones"]
for zid, z in zones.items():
    ws_name = z.get("worksheet", "")
    zc = z.get("zoneCommon", {})
    ztype = zc.get("zoneType", "")
    name = zc.get("name", "")
    is_map = z.get("presModelHolder", {}).get("visual", {}).get("isMap", None)
    print(f"zone {zid}: worksheet='{ws_name}' zoneType='{ztype}' name='{name}' isMap={is_map}")

zone 105: worksheet='Tabulasi SP2KP' zoneType='filter' name='Provinsi' isMap=None
zone 106: worksheet='Tabulasi SP2KP' zoneType='filter' name='Kabupaten/Kota' isMap=None
zone 128: worksheet='' zoneType='layout-flow' name='Horizontal Container' isMap=None
zone 129: worksheet='Tabulasi SP2KP' zoneType='filter' name='Komoditas' isMap=None
zone 138: worksheet='' zoneType='layout-flow' name='Horizontal Container' isMap=None
zone 17: worksheet='' zoneType='text' name='TABULASI HARGA SP2KP Sistem Pemantauan Pasar dan Kebutuhan Pokok (SP2KP)' isMap=None
zone 4: worksheet='' zoneType='layout-basic' name='Tiled' isMap=None
zone 47: worksheet='Tabulasi SP2KP' zoneType='viz' name='Tabulasi SP2KP' isMap=False
zone 48: worksheet='' zoneType='layout-basic' name='Tiled' isMap=None
zone 67: worksheet='' zoneType='paramctrl' name='Tanggal Awal' isMap=None
zone 70: worksheet='' zoneType='paramctrl' name='Tanggal Akhir' isMap=None


In [ ]:
ph_47 = zone_47["presModelHolder"]
print("Keys di presModelHolder zone 47:", list(ph_47.keys()))

Keys di presModelHolder zone 47: ['visual']


In [ ]:
import json
# print ukuran tiap field dulu, biar tau mana yang "berat" (kemungkinan berisi data)
for k, v in visual.items():
    v_str = json.dumps(v) if not isinstance(v, str) else v
    print(f"{k}: {len(str(v_str))} karakter")

hasVizArtTemplate: 5 karakter
hasVizExtension: 5 karakter
selectionList: 138 karakter
brushingSelectionList: 2 karakter
annotationList: 2 karakter
filtersJson: 31630 karakter
vizAltText: 64 karakter
geometryJson: 393 karakter
cacheUrlInfoJson: 186 karakter
sortIndicators: 28 karakter
isMap: 5 karakter
hasBackgroundImage: 5 karakter
floatingToolbarVisibility: 4 karakter
geographicSearchVisibility: 3 karakter
vizNavigationSetting: 5 karakter
isLayerControlButtonAllowed: 5 karakter
isLayerControlButtonEnabled: 4 karakter
wholeVizScrollThreshold: 3 karakter
defaultMapToolEnum: 21 karakter
hasModifiedAxes: 5 karakter
valid: 4 karakter
visualIdPresModel: 65 karakter
tooltipMode: 4 karakter
bgColor: 16 karakter
paneColor: 16 karakter
headerColor: 16 karakter
emptyHighlightFogAll: 5 karakter
totalMarks: 5 karakter
shouldShowStats: 5 karakter
showImageLimitToast: 5 karakter
selectedMarks: 1 karakter
numberOfColumns: 1 karakter
numberOfRows: 1 karakter
lastComputedCalculation: 0 karakter
animati

In [ ]:
import asyncio
import threading
import requests
from playwright.async_api import async_playwright

captured = {}

async def handle_response(response):
    if "bootstrapSession" in response.url:
        try:
            captured["raw"] = await response.text()
        except Exception:
            pass

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
        )
        page = await context.new_page()
        page.on("response", handle_response)

        await page.goto("https://sp2kp.kemendag.go.id/statistik/tabulasi-harga", wait_until="load", timeout=30000)
        await page.wait_for_timeout(10000)

        # ambil cookies dari browser context buat dipakai ulang di requests
        cookies = await context.cookies()
        captured["cookies"] = cookies

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

# ambil sessionId dari data bootstrap
import re, json
raw = captured["raw"]
first_semicolon = raw.index(";")
json_str = raw[first_semicolon+1:]
# ambil cuma sampai panjang yang tercantum di depan
length = int(raw[:first_semicolon])
json_str = raw[first_semicolon+1 : first_semicolon+1+length]
parsed = json.loads(json_str)
session_id = parsed["newSessionId"]
print("Session ID:", session_id)

Session ID: B4554B6BEFF749E996BB77ECA6E275F6-1:1


In [ ]:
import requests

# bangun session requests dari cookies yang diambil Playwright
s = requests.Session()
for c in captured["cookies"]:
    s.cookies.set(c["name"], c["value"], domain=c["domain"])

s.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Referer": "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
})

workbook_name = "TabulasiHargaSP2KP_17708080966730"
view_name = "TabulasiHarga"

export_url = (
    f"https://analitik.kemendag.go.id/vizql/w/{workbook_name}/v/{view_name}"
    f"/sessions/{session_id}/commands/tabsrv/export-crosstab-to-excel-server"
)

resp = s.post(export_url)

print("Status:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))
print("Content-Length:", len(resp.content))

if resp.status_code == 200 and len(resp.content) > 0:
    with open("tabulasi_harga.xlsx", "wb") as f:
        f.write(resp.content)
    print("Tersimpan sebagai tabulasi_harga.xlsx")

Status: 200
Content-Type: application/octet-stream;charset=UTF-8
Content-Length: 594
Tersimpan sebagai tabulasi_harga.xlsx


In [ ]:
import uuid
import random
import string

def gen_sheetdoc_id():
    return "{" + str(uuid.uuid4()).upper() + "}"

def gen_telemetry_id():
    chars = string.ascii_lowercase + string.digits
    return "".join(random.choice(chars) for _ in range(20))

payload = {
    "sheetdocId": gen_sheetdoc_id(),
    "sendNotifications": "true",
    "telemetryCommandId": gen_telemetry_id(),
}

print(payload)

resp = s.post(export_url, data=payload)

print("Status:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))
print("Content-Length:", len(resp.content))
print(resp.content[:300])

{'sheetdocId': '{F2609459-C294-4440-9DC5-C909DB196628}', 'sendNotifications': 'true', 'telemetryCommandId': 'whoppl6kvo1xqjdr64ql'}
Status: 500
Content-Type: application/json;charset=UTF-8
Content-Length: 493
b'\r\n{"timeStamp":"2026-07-02 09:22:53.341","errorResponseType":"Generic","errorType":"LogicException 2026-07-02 09:22:53.341 (akYt7G45V8C34AIvYvLmQwAAAc4,1:1):","errorExtras":"(akYt7G45V8C34AIvYvLmQwAAAc4,1:1)","details":["Internal Error - An unexpected error occurred and the operation could not be co'


In [ ]:
print(resp.content.decode('utf-8'))


{"timeStamp":"2026-07-02 09:22:53.341","errorResponseType":"Generic","errorType":"LogicException 2026-07-02 09:22:53.341 (akYt7G45V8C34AIvYvLmQwAAAc4,1:1):","errorExtras":"(akYt7G45V8C34AIvYvLmQwAAAc4,1:1)","details":["Internal Error - An unexpected error occurred and the operation could not be completed."],"message":"LogicException: Internal Error - An unexpected error occurred and the operation could not be completed.","type":"exception","communityLink":"https:\/\/public.tableau.com"}


In [ ]:
import asyncio
import threading
from playwright.async_api import async_playwright

download_path = {}

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            accept_downloads=True,
        )
        page = await context.new_page()

        try:
            await page.goto(
                "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
                wait_until="domcontentloaded",
                timeout=60000
            )
        except Exception as e:
            print("Gagal goto:", e)
            await page.screenshot(path="timeout_debug.png", full_page=True)
            await browser.close()
            return

        await page.wait_for_timeout(10000)
        await page.screenshot(path="before_click.png", full_page=True)

        download_btn = None
        selectors_to_try = [
            'button[title="Download"]',
            'div[title="Download"]',
            '[aria-label="Download"]',
            'button:has-text("Download")',
        ]
        for sel in selectors_to_try:
            try:
                el = page.locator(sel).first
                if await el.count() > 0:
                    download_btn = el
                    print("Ketemu selector:", sel)
                    break
            except Exception:
                continue

        if download_btn is None:
            print("Tombol download gak ketemu lewat selector biasa. Cek before_click.png")
            await browser.close()
            return

        await download_btn.click()
        await page.wait_for_timeout(1500)
        await page.screenshot(path="after_toolbar_click.png", full_page=True)

        crosstab_option = page.locator('text=Crosstab').first
        await crosstab_option.click()
        await page.wait_for_timeout(1500)
        await page.screenshot(path="crosstab_dialog.png", full_page=True)

        csv_radio = page.locator('text=CSV').first
        if await csv_radio.count() > 0:
            await csv_radio.click()

        async with page.expect_download(timeout=20000) as download_info:
            final_download_btn = page.locator('button:has-text("Download")').last
            await final_download_btn.click()

        download = await download_info.value
        # PATH RELATIF, bukan /tmp
        save_path = "tabulasi_harga_downloaded" + (".csv" if await csv_radio.count() > 0 else ".xlsx")
        await download.save_as(save_path)
        download_path["path"] = save_path
        print("File tersimpan di:", save_path)

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)
print(download_path)

Exception in thread Thread-12 (runner):
Traceback (most recent call last):
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Aufii\AppData\Local\Temp\ipykernel_12428\2751734652.py", line 85, in runner
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\asyncio\base_events.py", line 687, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "C:\Users\Aufii\AppData\Local\Temp\ipykernel_12428\2751734652.py", line 29, in run
  File "c:\Users\Aufii\AppData\Local\Programs\Python\Python312\Lib\site-packages\playwright\async_api\_generated.py", line 9739,

{}


In [ ]:
import asyncio
import threading
from playwright.async_api import async_playwright

download_path = {}
log = []

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            accept_downloads=True,
        )
        page = await context.new_page()

        await page.goto(
            "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
            wait_until="domcontentloaded",
            timeout=60000
        )
        log.append("goto OK")

        await page.wait_for_timeout(10000)
        log.append("wait 10s OK")

        # screenshot TANPA full_page, biar gak nunggu semua font
        try:
            await page.screenshot(path="before_click.png", timeout=10000)
            log.append("screenshot before_click OK")
        except Exception as e:
            log.append(f"screenshot gagal (dilewati): {e}")

        download_btn = None
        selectors_to_try = [
            'button[title="Download"]',
            'div[title="Download"]',
            '[aria-label="Download"]',
            'button:has-text("Download")',
        ]
        for sel in selectors_to_try:
            try:
                el = page.locator(sel).first
                if await el.count() > 0:
                    download_btn = el
                    log.append(f"Ketemu selector: {sel}")
                    break
            except Exception:
                continue

        if download_btn is None:
            log.append("Tombol download gak ketemu")
            await browser.close()
            return

        await download_btn.click()
        log.append("klik toolbar download OK")
        await page.wait_for_timeout(1500)

        crosstab_option = page.locator('text=Crosstab').first
        await crosstab_option.click()
        log.append("klik Crosstab OK")
        await page.wait_for_timeout(1500)

        csv_radio = page.locator('text=CSV').first
        if await csv_radio.count() > 0:
            await csv_radio.click()
            log.append("pilih CSV OK")

        async with page.expect_download(timeout=20000) as download_info:
            final_download_btn = page.locator('button:has-text("Download")').last
            await final_download_btn.click()
        log.append("klik Download final OK")

        download = await download_info.value
        save_path = "tabulasi_harga_downloaded" + (".csv" if await csv_radio.count() > 0 else ".xlsx")
        await download.save_as(save_path)
        download_path["path"] = save_path
        log.append(f"File tersimpan di: {save_path}")

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        except Exception as e:
            log.append(f"ERROR FATAL: {e}")
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("\n=== LOG ===")
for l in log:
    print(l)
print("\n", download_path)


=== LOG ===
goto OK
wait 10s OK
screenshot before_click OK
Tombol download gak ketemu

 {}


In [ ]:
import asyncio
import threading
from playwright.async_api import async_playwright

log = []
found_elements = []

async def scan_frame(frame, frame_label):
    try:
        elements = await frame.evaluate("""
            () => {
                const els = document.querySelectorAll('*');
                const results = [];
                for (const el of els) {
                    const cls = (el.className && typeof el.className === 'string') ? el.className : '';
                    const dataAttrs = Array.from(el.attributes)
                        .filter(a => a.name.startsWith('data-'))
                        .map(a => `${a.name}=${a.value}`)
                        .join(' ');
                    const combined = (cls + ' ' + dataAttrs).toLowerCase();
                    if (combined.includes('download') || combined.includes('export')) {
                        results.push({
                            tag: el.tagName,
                            class: cls,
                            dataAttrs: dataAttrs,
                            id: el.id,
                        });
                    }
                }
                return results;
            }
        """)
        for e in elements:
            e["frame"] = frame_label
            found_elements.append(e)
    except Exception as e:
        log.append(f"gagal scan frame {frame_label}: {e}")

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            accept_downloads=True,
        )
        page = await context.new_page()

        await page.goto(
            "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
            wait_until="domcontentloaded",
            timeout=60000
        )
        await page.wait_for_timeout(10000)

        # scan main page + semua iframe
        await scan_frame(page, "main")
        for i, f in enumerate(page.frames):
            if f != page.main_frame:
                await scan_frame(f, f"iframe-{i}: {f.url[:80]}")

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        except Exception as e:
            log.append(f"ERROR: {e}")
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("=== LOG ===")
for l in log:
    print(l)

print(f"\n=== ELEMEN DOWNLOAD/EXPORT ({len(found_elements)}) ===")
for e in found_elements[:30]:
    print(e)

=== LOG ===

=== ELEMEN DOWNLOAD/EXPORT (2) ===
{'tag': 'BUTTON', 'class': 'dropdown-button fx8zm5x', 'dataAttrs': 'data-eventutils-optout=true data-tb-test-id=viz-viewer-toolbar-button-download data-toolbar-icon-id=download', 'id': 'download', 'frame': 'iframe-1: https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/Tabulasi'}
{'tag': 'svg', 'class': '', 'dataAttrs': 'data-tb-test-id=tb-icons-DownloadBaseIcon', 'id': '', 'frame': 'iframe-1: https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/Tabulasi'}


# final scrape code

In [6]:
import asyncio
import threading
from playwright.async_api import async_playwright

log = []
download_path = {}

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            accept_downloads=True,
        )
        page = await context.new_page()

        await page.goto(
            "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
            wait_until="domcontentloaded",
            timeout=60000
        )
        await page.wait_for_timeout(10000)

        # cari frame yang cocok berdasarkan URL (bukan page.locator biasa)
        target_frame = None
        for f in page.frames:
            if "analitik.kemendag.go.id/views/" in f.url:
                target_frame = f
                break

        if target_frame is None:
            log.append("Frame Tableau gak ketemu")
            await browser.close()
            return

        log.append(f"Frame ketemu: {target_frame.url[:80]}")

        # klik tombol download di dalam frame itu
        download_btn = target_frame.locator("#download")
        await download_btn.click()
        log.append("klik tombol download OK")
        await page.wait_for_timeout(1500)

        # cari opsi Crosstab yang muncul (biasanya dropdown menu di frame yang sama)
        crosstab_option = target_frame.locator("text=Crosstab").first
        await crosstab_option.click()
        log.append("klik Crosstab OK")
        await page.wait_for_timeout(1500)

        # dialog "Download Crosstab" -> pilih CSV
        csv_radio = target_frame.locator("text=CSV").first
        if await csv_radio.count() > 0:
            await csv_radio.click()
            log.append("pilih CSV OK")

        # klik tombol Download final di dialog, tangkep file-nya
        async with page.expect_download(timeout=20000) as download_info:
            final_btn = target_frame.locator('button:has-text("Download")').last
            await final_btn.click()
        log.append("klik Download final OK")

        download = await download_info.value
        save_path = "tabulasi_harga_downloaded.csv"
        await download.save_as(save_path)
        download_path["path"] = save_path
        log.append(f"File tersimpan di: {save_path}")

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        except Exception as e:
            log.append(f"ERROR: {e}")
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("=== LOG ===")
for l in log:
    print(l)
print(download_path)

=== LOG ===
Frame ketemu: https://analitik.kemendag.go.id/views/TabulasiHargaSP2KP_17708080966730/Tabulasi
klik tombol download OK
klik Crosstab OK
pilih CSV OK
klik Download final OK
File tersimpan di: tabulasi_harga_downloaded.csv
{'path': 'tabulasi_harga_downloaded.csv'}


In [9]:
import asyncio
import threading
from playwright.async_api import async_playwright

log = []
download_path = {}

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            accept_downloads=True,
        )
        page = await context.new_page()

        await page.goto(
            "https://sp2kp.kemendag.go.id/statistik/tabulasi-harga",
            wait_until="domcontentloaded",
            timeout=60000
        )
        await page.wait_for_timeout(10000)

        target_frame = None
        for f in page.frames:
            if "analitik.kemendag.go.id/views/" in f.url:
                target_frame = f
                break

        if target_frame is None:
            log.append("Frame Tableau gak ketemu")
            await browser.close()
            return

        download_btn = target_frame.locator("#download")
        await download_btn.click()
        await page.wait_for_timeout(1500)

        crosstab_option = target_frame.locator("text=Crosstab").first
        await crosstab_option.click()
        await page.wait_for_timeout(1500)

        csv_radio = target_frame.locator("text=CSV").first
        if await csv_radio.count() > 0:
            await csv_radio.click()

        async with page.expect_download(timeout=20000) as download_info:
            final_btn = target_frame.locator('button:has-text("Download")').last
            await final_btn.click()

        download = await download_info.value
        save_path = "../data/raw/groupF/9_ekonomi_harga_pangan_harian.csv"
        await download.save_as(save_path)
        download_path["path"] = save_path
        log.append(f"Tersimpan: {save_path}")

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        except Exception as e:
            log.append(f"ERROR: {e}")
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)
print(log)
print(download_path)

['Tersimpan: ../data/raw/groupF/9_ekonomi_harga_pangan_harian.csv']
{'path': '../data/raw/groupF/9_ekonomi_harga_pangan_harian.csv'}


In [10]:
import pandas as pd
df = pd.read_csv("../data/raw/groupF/9_ekonomi_harga_pangan_harian.csv", sep="\t", encoding="utf-16")
print(df.shape)
print(df.columns.tolist()[-10:])

(7784, 70)
['22/06/2026', '23/06/2026', '24/06/2026', '25/06/2026', '26/06/2026', '29/06/2026', '30/06/2026', '01/07/2026', '02/07/2026', '03/07/2026']
